# Chapter 59: Performance Optimization & Deployment
## Multimedia & Computer Vision Learning Hub — ITERA
**Author:** Aji Pamungkas Tri Nurcahyo  
**Objective:** Menganalisis alokasi anggaran waktu frame (*frame-time budgeting*) untuk sistem visi waktu-nyata 60 FPS, memformulasikan batas teoritis percepatan sistem menggunakan Hukum Amdahl (*Amdahl's Law*), mengidentifikasi titik hambat komputasi (*bottleneck analysis*), serta memvalidasi teknik optimasi bundel statis dan distribusi web berkinerja tinggi.

### 1. Landasan Matematis: Anggaran Frame & Hukum Amdahl
Batas Anggaran Frame (Frame Budget):
$$T_{\text{budget}} = \frac{1000}{F_{\text{target}}} \quad (16.67 \text{ ms untuk 60 FPS})$$

Dekomposisi Latensi Pipa:
$$T_{\text{total}} = \sum_{k=1}^K T_k, \quad T_{\text{headroom}} = T_{\text{budget}} - T_{\text{total}}$$

Formulasi Hukum Amdahl (Amdahl's Law):
$$S_{\text{overall}} = \frac{1}{(1 - p) + \frac{p}{s}}$$
Di mana $p$ adalah proporsi durasi tahapan awal, dan $s$ adalah faktor percepatan tahapan tersebut.

In [ ]:
# 2. Setup Lingkungan & Dual-Engine Check
import numpy as np
import sys

print(f"Python Version: {sys.version}")
print(f"NumPy Version: {np.__version__}")

HAS_TORCH = False
try:
    import torch
    HAS_TORCH = True
    print(f"PyTorch: {torch.__version__} (Device: {'cuda' if torch.cuda.is_available() else 'cpu'})")
except ImportError:
    print("INFO: PyTorch tidak terpasang di host lokal. Beroperasi dalam Mode Analisis Rekayasa Kinerja NumPy murni.")

In [ ]:
# 3. Validasi Numerik Skenario A: Dekomposisi Anggaran Latensi 60 FPS
stages = {'capture': 2.5, 'preprocess': 1.5, 'inference': 8.0, 'postprocess': 1.0}
t_total = sum(stages.values())
t_budget = 1000.0 / 60.0
headroom = t_budget - t_total
max_fps = 1000.0 / t_total

bottleneck_stage = max(stages, key=stages.get)
p_bottleneck = stages[bottleneck_stage] / t_total

print(f"Total Latensi: {t_total:.2f} ms | Budget: {t_budget:.2f} ms | Headroom: {headroom:+.2f} ms")
print(f"Achievable FPS: {max_fps:.2f} FPS")
print(f"Bottleneck: '{bottleneck_stage}' ({p_bottleneck*100:.2f}% dari total waktu)")

assert t_total == 13.0
np.testing.assert_allclose(t_budget, 16.666667, atol=1e-4)
np.testing.assert_allclose(headroom, 3.666667, atol=1e-4)
assert headroom > 0, "Sistem memenuhi batas 60 FPS"
assert bottleneck_stage == 'inference'
np.testing.assert_allclose(p_bottleneck, 8.0 / 13.0, atol=1e-4)
print("VALIDASI DEKOMPOSISI ANGGARAN: Lulus sempurna!")

In [ ]:
# 4. Validasi Numerik Skenario B: Penerapan Hukum Amdahl (Speedup 4x)
def amdahl_speedup(p, s):
    return 1.0 / ((1.0 - p) + (p / s))

p_inf = 8.0 / 13.0
s_inf = 4.0
s_overall = amdahl_speedup(p_inf, s_inf)

# Verifikasi Langsung
new_total = 2.5 + 1.5 + (8.0 / 4.0) + 1.0 # 7.0 ms
s_direct = t_total / new_total
new_fps = 1000.0 / new_total

print(f"Amdahl Speedup S_overall: {s_overall:.4f}x")
print(f"Direct Speedup (13.0/7.0): {s_direct:.4f}x")
print(f"Throughput Baru: {new_fps:.2f} FPS (Durasi: {new_total:.2f} ms)")

np.testing.assert_allclose(s_overall, 13.0 / 7.0, atol=1e-4)
np.testing.assert_allclose(s_direct, 13.0 / 7.0, atol=1e-4)
assert new_total == 7.0
np.testing.assert_allclose(new_fps, 142.857143, atol=1e-3)
print("VALIDASI HUKUM AMDAHL: Lulus matematis!")

In [ ]:
# 5. Analisis Sensitivitas Amdahl & Profiling Bundel Statis Web
p_values = [0.2, 0.5, 0.8, 0.95]
s_values = [2.0, 4.0, 8.0, 16.0]

print(f"{'Proporsi (p)':<14} | " + " | ".join([f"s={s:<4.0f}" for s in s_values]))
print("-" * 50)
for p_val in p_values:
    row = [f"{amdahl_speedup(p_val, s_val):<6.2f}x" for s_val in s_values]
    print(f"p = {p_val:<10.2f} | " + " | ".join(row))

# Profiling Bundel Statis
raw_vendor_kb, raw_app_kb, raw_css_kb = 150.0, 50.0, 20.0
raw_total = raw_vendor_kb + raw_app_kb + raw_css_kb
gzip_total = raw_total * 0.30
print(f"\nUkuran Bundel Statis: {raw_total:.1f} KB -> Gzip (30%): {gzip_total:.1f} KB")
assert raw_total == 220.0
assert gzip_total == 66.0
print("ANALISIS SENSITIVITAS & BUNDEL: Selesai.")

### 6. Rangkuman & Pembahasan Akademik
1. **Batas Teoretis Hukum Amdahl**: Percepatan sistem dibatasi oleh fraksi yang tidak dapat dipercepat ($1 - p$). Selalu prioritaskan optimasi pada bottleneck terbesar ($p > 0.5$).
2. **Disiplin Anggaran Frame**: Menjaga latensi total di bawah 16.67 ms menjamin interaktivitas 60 FPS tanpa jeda pada aplikasi visi peramban.
3. **Distribusi Statis Efisien**: Melalui pemisahan kode (*code splitting*) dan kompresi Gzip/Brotli, seluruh aset platform pembelajaran ini dapat dimuat dalam waktu kurang dari 1 detik bahkan di jaringan seluler 3G.